## Hazard × Span Intersection Pipeline
**Task:** P9-IMPL-HAZARD-SPAN-XWALK | **Priority:** HIGH | **Agent:** @data-engineer

Spatial join between `geographic_hazard_event` (50K+ rows) and `physical_resource_span` (10K rows).
Produces `hazard_span_intersection` — the silver crosswalk that bridges hazard signals to physical fiber spans for resilience scoring.

**Algorithm:**
1. Bounding-box pre-filter (lat/lon ± radius_km → degrees)
2. Haversine distance calculation
3. Proximity bands: `direct_impact` (<1km), `near_field` (1–5km), `buffer_zone` (5–radius_km)
4. SHA2 deterministic intersection_id
5. Affected service count from `service_path`

**Source:** Spec in notebook `hazard_schema_design` cell 7. Lakelink Fiber PRI.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS cdm_tmforum.provable_resilience_intelligence.hazard_span_intersection (
  intersection_id       STRING        NOT NULL  COMMENT 'Deterministic ID: SHA2(hazard_source || hazard_source_id || span_id, 256)',
  hazard_source         STRING        NOT NULL  COMMENT 'Staging source: noaa_weather | usgs_earthquake | usgs_flood | nifc_wildfire | 811_dig_ticket',
  hazard_source_id      STRING        NOT NULL  COMMENT 'PK from staging table (alert_id, event_id, warning_id, irwin_id, ticket_number)',
  hazard_type           STRING        NOT NULL  COMMENT 'Normalized type: severe_weather | earthquake | flood | wildfire | dig_ticket',
  hazard_severity       STRING        NOT NULL  COMMENT 'Normalized severity: critical | high | medium | low (mapped from source-native scales)',
  hazard_lat            DOUBLE        NOT NULL  COMMENT 'Hazard centroid/epicenter latitude',
  hazard_lon            DOUBLE        NOT NULL  COMMENT 'Hazard centroid/epicenter longitude',
  hazard_radius_km      DOUBLE        NOT NULL  COMMENT 'Impact radius used for this intersection',
  hazard_start          TIMESTAMP     NOT NULL  COMMENT 'Hazard event start time (UTC)',
  hazard_end            TIMESTAMP               COMMENT 'Hazard event end time (UTC). NULL if ongoing.',
  span_id               STRING        NOT NULL  COMMENT 'FK to physical_resource_span.span_id',
  span_midpoint_lat     DOUBLE        NOT NULL  COMMENT 'Span midpoint latitude (denormalized for query performance)',
  span_midpoint_lon     DOUBLE        NOT NULL  COMMENT 'Span midpoint longitude (denormalized for query performance)',
  distance_km           DOUBLE        NOT NULL  COMMENT 'Haversine distance from hazard centroid to span midpoint in km',
  is_within_radius      BOOLEAN       NOT NULL  COMMENT 'TRUE if distance_km <= hazard_radius_km',
  proximity_band        STRING        NOT NULL  COMMENT 'Distance band: direct_impact (<1km) | near_field (1-5km) | buffer_zone (5-radius_km)',
  affected_service_ids  STRING                  COMMENT 'Comma-separated service_ids routed through this span',
  affected_service_cnt  INT                     COMMENT 'Count of services routed through this span',
  computed_at           TIMESTAMP     NOT NULL  COMMENT 'UTC timestamp when this intersection was computed',
  batch_id              STRING        NOT NULL  COMMENT 'Processing batch identifier for incremental refresh'
)
COMMENT 'Silver: Hazard-to-span spatial intersection crosswalk. One row per (hazard, span) pair within buffer radius. Bridges staging tables to physical_resource_span for resilience scoring. Lakelink Fiber PRI.'
TBLPROPERTIES (
  'quality.pipeline' = 'silver',
  'refresh.cadence' = 'event-driven',
  'sid.concept' = 'Common.GeographicPlace x Resource.PhysicalResourceSpan'
);

In [0]:
%sql
-- ==========================================================================
-- HAZARD x SPAN SPATIAL JOIN
-- Algorithm: Bounding-box pre-filter → Haversine → Proximity bands
-- Radius conversion: 1° lat ≈ 111.32km; 1° lon ≈ 111.32km * COS(lat)
-- Haversine formula for great-circle distance on WGS84 sphere
-- ==========================================================================

INSERT OVERWRITE cdm_tmforum.provable_resilience_intelligence.hazard_span_intersection
WITH 
-- Map source field to staging source name per spec
hazard_source_map AS (
  SELECT 
    hazard_id,
    hazard_type,
    severity AS hazard_severity,
    latitude AS hazard_lat,
    longitude AS hazard_lon,
    radius_km AS hazard_radius_km,
    event_start AS hazard_start,
    event_end AS hazard_end,
    -- Map hazard_type → hazard_source per spec
    CASE hazard_type
      WHEN 'severe_weather' THEN 'noaa_weather'
      WHEN 'earthquake'     THEN 'usgs_earthquake'
      WHEN 'flood'          THEN 'usgs_flood'
      WHEN 'wildfire'       THEN 'nifc_wildfire'
      WHEN 'dig_ticket'     THEN '811_dig_ticket'
      ELSE LOWER(REPLACE(source, ' ', '_'))
    END AS hazard_source,
    hazard_id AS hazard_source_id,
    -- Bounding-box deltas in degrees (for pre-filter)
    radius_km / 111.32 AS delta_lat_deg,
    radius_km / (111.32 * COS(RADIANS(latitude))) AS delta_lon_deg
  FROM cdm_tmforum.provable_resilience_intelligence.geographic_hazard_event
  WHERE latitude IS NOT NULL AND longitude IS NOT NULL AND radius_km > 0
),

-- Bounding-box pre-filter: only pairs where span midpoint is within lat/lon box
bbox_candidates AS (
  SELECT 
    h.*,
    s.span_id,
    s.midpoint_lat AS span_midpoint_lat,
    s.midpoint_lon AS span_midpoint_lon
  FROM hazard_source_map h
  INNER JOIN cdm_tmforum.provable_resilience_intelligence.physical_resource_span s
    ON s.midpoint_lat BETWEEN (h.hazard_lat - h.delta_lat_deg) AND (h.hazard_lat + h.delta_lat_deg)
   AND s.midpoint_lon BETWEEN (h.hazard_lon - h.delta_lon_deg) AND (h.hazard_lon + h.delta_lon_deg)
   AND s.midpoint_lat IS NOT NULL
   AND s.midpoint_lon IS NOT NULL
),

-- Haversine distance calculation
haversine_calc AS (
  SELECT 
    *,
    -- Haversine formula: d = 2R * arcsin(sqrt(sin²(Δlat/2) + cos(lat1)*cos(lat2)*sin²(Δlon/2)))
    2 * 6371.0 * ASIN(SQRT(
      POWER(SIN(RADIANS(span_midpoint_lat - hazard_lat) / 2), 2) +
      COS(RADIANS(hazard_lat)) * COS(RADIANS(span_midpoint_lat)) *
      POWER(SIN(RADIANS(span_midpoint_lon - hazard_lon) / 2), 2)
    )) AS distance_km
  FROM bbox_candidates
),

-- Filter to within radius + classify proximity bands
proximity_classified AS (
  SELECT 
    *,
    (distance_km <= hazard_radius_km) AS is_within_radius,
    CASE 
      WHEN distance_km < 1.0 THEN 'direct_impact'
      WHEN distance_km < 5.0 THEN 'near_field'
      ELSE 'buffer_zone'
    END AS proximity_band
  FROM haversine_calc
  WHERE distance_km <= hazard_radius_km  -- Only keep pairs within the hazard radius
),

-- BUG-QA-060: resolve affected services at CONDUIT level.
-- service_path records one representative segment per conduit traversal, so a span-level join
-- under-counts exposure. Resolve span -> conduit_id -> every service whose path crosses that conduit.
conduit_services AS (
  SELECT s.conduit_id, ARRAY_SORT(COLLECT_SET(p.service_id)) AS svcs
  FROM cdm_tmforum.provable_resilience_intelligence.service_path p
  JOIN cdm_tmforum.provable_resilience_intelligence.physical_resource_span s ON p.span_id = s.span_id
  GROUP BY s.conduit_id
),
span_services AS (
  SELECT 
    s.span_id,
    CONCAT_WS(',', c.svcs) AS affected_service_ids,
    CAST(COALESCE(SIZE(c.svcs), 0) AS INT) AS affected_service_cnt
  FROM cdm_tmforum.provable_resilience_intelligence.physical_resource_span s
  LEFT JOIN conduit_services c ON s.conduit_id = c.conduit_id
)

-- Final output: deterministic ID, all columns per spec
SELECT 
  SHA2(CONCAT(p.hazard_source, '|', p.hazard_source_id, '|', p.span_id), 256) AS intersection_id,
  p.hazard_source,
  p.hazard_source_id,
  p.hazard_type,
  p.hazard_severity,
  p.hazard_lat,
  p.hazard_lon,
  p.hazard_radius_km,
  p.hazard_start,
  p.hazard_end,
  p.span_id,
  p.span_midpoint_lat,
  p.span_midpoint_lon,
  ROUND(p.distance_km, 4) AS distance_km,
  p.is_within_radius,
  p.proximity_band,
  sv.affected_service_ids,
  sv.affected_service_cnt,
  CURRENT_TIMESTAMP() AS computed_at,
  CONCAT('batch-', DATE_FORMAT(CURRENT_TIMESTAMP(), 'yyyyMMdd-HHmmss')) AS batch_id
FROM proximity_classified p
LEFT JOIN span_services sv ON p.span_id = sv.span_id
ORDER BY p.hazard_severity, p.distance_km;

In [0]:
%sql
-- ==========================================================================
-- VALIDATION: Row counts, proximity distribution, and quality checks
-- ==========================================================================

SELECT '1. Total intersections' AS metric, CAST(COUNT(*) AS STRING) AS value
FROM cdm_tmforum.provable_resilience_intelligence.hazard_span_intersection

UNION ALL
SELECT '2. Distinct hazards with span hits', CAST(COUNT(DISTINCT hazard_source_id) AS STRING)
FROM cdm_tmforum.provable_resilience_intelligence.hazard_span_intersection

UNION ALL
SELECT '3. Distinct spans affected', CAST(COUNT(DISTINCT span_id) AS STRING)
FROM cdm_tmforum.provable_resilience_intelligence.hazard_span_intersection

UNION ALL
SELECT '4. Avg distance_km', CAST(ROUND(AVG(distance_km), 2) AS STRING)
FROM cdm_tmforum.provable_resilience_intelligence.hazard_span_intersection

UNION ALL
SELECT '5. NULL intersection_id', CAST(COUNT(*) AS STRING)
FROM cdm_tmforum.provable_resilience_intelligence.hazard_span_intersection WHERE intersection_id IS NULL

UNION ALL
SELECT '6. is_within_radius = TRUE', CAST(COUNT(*) AS STRING)
FROM cdm_tmforum.provable_resilience_intelligence.hazard_span_intersection WHERE is_within_radius = TRUE;

In [0]:
%sql
-- Proximity band × hazard type distribution
SELECT 
  hazard_type,
  proximity_band,
  COUNT(*) AS intersection_cnt,
  COUNT(DISTINCT span_id) AS unique_spans,
  COUNT(DISTINCT hazard_source_id) AS unique_hazards,
  ROUND(AVG(distance_km), 2) AS avg_distance_km,
  ROUND(AVG(affected_service_cnt), 1) AS avg_services_affected,
  SUM(affected_service_cnt) AS total_service_exposures
FROM cdm_tmforum.provable_resilience_intelligence.hazard_span_intersection
GROUP BY hazard_type, proximity_band
ORDER BY hazard_type, 
  CASE proximity_band WHEN 'direct_impact' THEN 1 WHEN 'near_field' THEN 2 ELSE 3 END;

In [0]:
%sql
-- Severity distribution by hazard source — feeds into resilience_score_snapshot
SELECT 
  hazard_source,
  hazard_severity,
  COUNT(*) AS intersections,
  COUNT(DISTINCT span_id) AS spans_at_risk,
  ROUND(SUM(affected_service_cnt), 0) AS total_service_exposures,
  ROUND(AVG(distance_km), 2) AS avg_distance_km,
  ROUND(MIN(distance_km), 3) AS closest_km
FROM cdm_tmforum.provable_resilience_intelligence.hazard_span_intersection
GROUP BY hazard_source, hazard_severity
ORDER BY hazard_source, 
  CASE hazard_severity WHEN 'critical' THEN 1 WHEN 'high' THEN 2 WHEN 'medium' THEN 3 ELSE 4 END;

In [0]:
%sql
-- FK integrity: hazard_span_intersection.span_id → physical_resource_span.span_id
SELECT 
  'span_id FK' AS fk_check,
  COUNT(*) AS total_rows,
  COUNT(s.span_id) AS resolved,
  COUNT(*) - COUNT(s.span_id) AS orphans,
  ROUND(100.0 * COUNT(s.span_id) / COUNT(*), 2) AS pct_resolved
FROM cdm_tmforum.provable_resilience_intelligence.hazard_span_intersection h
LEFT JOIN cdm_tmforum.provable_resilience_intelligence.physical_resource_span s
  ON h.span_id = s.span_id

UNION ALL

-- FK: hazard_source_id → geographic_hazard_event.hazard_id
SELECT 
  'hazard_id FK',
  COUNT(*),
  COUNT(g.hazard_id),
  COUNT(*) - COUNT(g.hazard_id),
  ROUND(100.0 * COUNT(g.hazard_id) / COUNT(*), 2)
FROM cdm_tmforum.provable_resilience_intelligence.hazard_span_intersection h
LEFT JOIN cdm_tmforum.provable_resilience_intelligence.geographic_hazard_event g
  ON h.hazard_source_id = g.hazard_id;